# Assignment 2 – Smart Document Scanner (Self-contained Colab)
This notebook implements the classical methods required by Tasks 1–10. It generates ten synthetic document pages in memory, so no image upload or external download is required. **Important:** these are controlled synthetic test fixtures, not real library photographs. Replace the generator/input list with actual photographed pages and verified ground-truth masks before reporting results as real-world measurements.

Runtime: Python + OpenCV + NumPy + Matplotlib. The notebook includes visual outputs, metrics, timing, and a basic integrated scanner. Run all cells from top to bottom.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, time, pandas as pd, math, os, textwrap
from IPython.display import display
np.random.seed(42)
print("OpenCV:", cv2.__version__)

In [ ]:
def make_document(seed=0, h=700, w=500, shadow=True, stain=True):
    rng=np.random.default_rng(seed)
    img=np.full((h,w),238,np.float32)
    # uneven illumination / shadow
    yy,xx=np.mgrid[0:h,0:w]
    illum=1.0- (0.22*np.exp(-((xx-w*0.82)**2)/(2*(w*.22)**2)) if shadow else 0)
    img*=illum
    # faint horizontal scanner bands
    img += 1.5*np.sin(2*np.pi*yy/6.0)
    # printed lines and text-like strokes; save exact foreground mask
    gt=np.zeros((h,w),np.uint8)
    cv2.rectangle(gt,(45,45),(w-45,h-45),0,1)
    for y in range(100,h-65,58):
        cv2.line(gt,(65,y),(w-65,y),255,1)
        for j in range(4):
            x=70+j*95
            cv2.rectangle(gt,(x,y+12),(min(x+65,w-65),y+19),255,-1)
            cv2.circle(gt,(x+5,y+32),2,255,-1)
    # render dark ink from mask
    img[gt>0]=np.minimum(img[gt>0],rng.uniform(65,100))
    # stain region
    stainmask=np.zeros((h,w),np.uint8)
    if stain:
        cv2.ellipse(stainmask,(int(w*.78),int(h*.72)),(35,24),-18,0,360,255,-1)
        img[stainmask>0]=np.minimum(img[stainmask>0],185)
    # impulse dust and camera grain
    noisy=img+rng.normal(0,5.5,(h,w))
    salt=rng.random((h,w))<0.003
    pepper=rng.random((h,w))<0.003
    noisy[salt]=255; noisy[pepper]=0
    return np.uint8(np.clip(noisy,0,255)), np.uint8(gt>0)*255, stainmask

pages=[make_document(i, shadow=(i%3==0), stain=(i%2==0)) for i in range(10)]
print("Generated",len(pages),"synthetic document pages.")
plt.figure(figsize=(7,5)); plt.imshow(pages[0][0],cmap='gray'); plt.title("Synthetic input page"); plt.axis("off"); plt.show()

## Utility functions and metrics

In [ ]:
def elapsed(fn, *args, repeats=3, **kwargs):
    vals=[]
    for _ in range(repeats):
        t=time.perf_counter(); out=fn(*args,**kwargs); vals.append(time.perf_counter()-t)
    return out, float(np.mean(vals))
def dice(a,b):
    a=np.asarray(a)>0; b=np.asarray(b)>0
    den=a.sum()+b.sum()
    return 1.0 if den==0 else float(2*np.logical_and(a,b).sum()/den)
def psnr(a,b):
    return float(cv2.PSNR(np.uint8(a),np.uint8(b)))
def entropy(im):
    hist=np.bincount(np.asarray(im).ravel(),minlength=256).astype(float); p=hist[hist>0]/hist.sum()
    return float(-(p*np.log2(p)).sum())
def contrast(im, mask):
    im=np.asarray(im)
    ink=im[mask>0]; paper=im[mask==0]
    return float(paper.mean()-ink.mean()) if len(ink) and len(paper) else 0.0
def show_images(images,titles,cmap='gray',figsize=(13,4)):
    plt.figure(figsize=figsize)
    for i,(im,title) in enumerate(zip(images,titles),1):
        plt.subplot(1,len(images),i); plt.imshow(im,cmap=cmap); plt.title(title); plt.axis('off')
    plt.tight_layout(); plt.show()


## Task 1 – Point transformations, histograms and bit planes

In [ ]:
def negative_log(im):
    x=im.astype(np.float32); log=np.log1p(x); return np.uint8(255*(log/log.max()))
def gamma_transform(im,gamma=1.25):
    lut=np.uint8(np.clip((np.arange(256)/255.0)**gamma*255,0,255)); return cv2.LUT(im,lut)
def piecewise(im,r1=90,r2=180):
    lut=np.interp(np.arange(256),[0,r1,r2,255],[0,0,255,255]).astype(np.uint8); return cv2.LUT(im,lut)
im,gt,sm=pages[0]
trans=[negative_log(im),gamma_transform(im,1.25),piecewise(im)]
show_images([im]+trans,["Input","Negative/log","Gamma 1.25","Piecewise"])
plt.figure(figsize=(7,4))
for name,out in zip(["input","negative/log","gamma","piecewise"],[im]+trans):
    plt.hist(out.ravel(),bins=64,alpha=.35,label=name)
plt.legend(); plt.title("Intensity histograms"); plt.show()
contrast_rows=[]
for name,fn in [("Negative/log",negative_log),("Gamma",lambda x:gamma_transform(x,1.25)),("Piecewise",piecewise)]:
    vals=[]; times=[]
    for x,g,_ in pages:
        y,t=elapsed(fn,x); vals.append(contrast(y,g)); times.append(t)
    contrast_rows.append([name,np.mean(vals),np.mean(times)*1000])
display(pd.DataFrame(contrast_rows,columns=["Method","Mean ink-paper separation","Mean time (ms)"]))
# bit planes
show_images([((im>>b)&1)*255 for b in range(7,-1,-1)],[f"Bit {b}" for b in range(7,-1,-1)],figsize=(14,4))

## Task 2 – Histogram equalization, matching and CLAHE

In [ ]:
ref=pages[1][0]
def hist_match(src,ref):
    s,c=np.unique(src.ravel(),return_counts=True); r,rc=np.unique(ref.ravel(),return_counts=True)
    sc=np.cumsum(c).astype(float); sc/=sc[-1]
    rcdf=np.cumsum(rc).astype(float); rcdf/=rcdf[-1]
    lut=np.interp(sc,rcdf,r).clip(0,255).astype(np.uint8)
    full=np.zeros(256,np.uint8); full[s]=lut
    return full[src]
def clahe(im,clip=2.0,grid=(8,8)):
    return cv2.createCLAHE(clipLimit=clip,tileGridSize=grid).apply(im)
im=pages[0][0]
outs=[cv2.equalizeHist(im),hist_match(im,ref),clahe(im)]
show_images([im]+outs,["Input","Global equalization","Histogram match","CLAHE"])
rows=[]
for name,fn in [("Global",cv2.equalizeHist),("Match",lambda x:hist_match(x,ref)),("CLAHE",clahe)]:
    vals=[]; ents=[]; ts=[]
    for x,g,_ in pages:
        y,t=elapsed(fn,x); vals.append(contrast(y,g)); ents.append(entropy(y)); ts.append(t)
    rows.append([name,np.mean(vals),np.mean(ents),np.mean(ts)*1000])
display(pd.DataFrame(rows,columns=["Method","Mean contrast","Entropy (bits)","Time (ms)"]))
plt.figure(figsize=(8,4))
for name,out in zip(["Input","Global","Match","CLAHE"],[im]+outs):
    h=np.bincount(out.ravel(),minlength=256); c=np.cumsum(h)/h.sum(); plt.plot(c,label=name)
plt.legend(); plt.title("Cumulative distribution functions"); plt.show()

## Task 3 – Spatial smoothing and PSNR

In [ ]:
def box3(x): return cv2.blur(x,(3,3))
def gauss3(x): return cv2.GaussianBlur(x,(3,3),0.8)
def median3(x): return cv2.medianBlur(x,3)
im,gt,_=pages[0]
outs=[box3(im),gauss3(im),median3(im)]
show_images([im]+outs,["Input","Box 3x3","Gaussian 3x3","Median 3x3"])
rows=[]
for k in [3,5,7]:
    for name,fn in [("Box",lambda x,k=k:cv2.blur(x,(k,k))),("Gaussian",lambda x,k=k:cv2.GaussianBlur(x,(k,k),0)),("Median",lambda x,k=k:cv2.medianBlur(x,k))]:
        scores=[]; ts=[]
        for x,_,_ in pages:
            clean=cv2.medianBlur(x,7) # proxy reference only for controlled synthetic fixture
            y,t=elapsed(fn,x); scores.append(psnr(clean,y)); ts.append(t)
        rows.append([name,k,np.mean(scores),np.mean(ts)*1000])
display(pd.DataFrame(rows,columns=["Filter","Kernel","PSNR vs proxy (dB)","Time (ms)"]))
# intensity profile
row=150; plt.figure(figsize=(9,3)); plt.plot(im[row],label="input"); plt.plot(median3(im)[row],label="median"); plt.legend(); plt.title("Horizontal intensity profile"); plt.show()


## Task 4 – Sharpening

In [ ]:
def lap_sharp(x):
    lap=cv2.Laplacian(x,cv2.CV_32F,ksize=3); return np.uint8(np.clip(x.astype(np.float32)-0.35*lap,0,255))
def unsharp(x):
    blur=cv2.GaussianBlur(x,(5,5),1.0); return cv2.addWeighted(x,1.5,blur,-0.5,0)
def sobel_sharp(x):
    sx=cv2.Sobel(x,cv2.CV_32F,1,0,ksize=3); sy=cv2.Sobel(x,cv2.CV_32F,0,1,ksize=3)
    mag=cv2.magnitude(sx,sy); return np.uint8(np.clip(x.astype(np.float32)+0.15*mag,0,255))
im=pages[0][0]; outs=[lap_sharp(im),unsharp(im),sobel_sharp(im)]
show_images([im]+outs,["Input","Laplacian","Unsharp","Sobel gradient"])
display(pd.DataFrame([[n,cv2.Laplacian(o,cv2.CV_64F).var(),np.std(o[20:50,20:50])] for n,o in zip(["Laplacian","Unsharp","Sobel"],outs)],columns=["Method","Variance of Laplacian","Blank-region std (illustrative ROI)"]))

## Task 5 – Frequency-domain low-pass filters

In [ ]:
def freq_filter(im,kind='gaussian',d0=35,order=2,high=False):
    h,w=im.shape; pad=cv2.copyMakeBorder(im, h//2,h//2,w//2,w//2,cv2.BORDER_REFLECT)
    F=np.fft.fftshift(np.fft.fft2(pad)); yy,xx=np.mgrid[0:pad.shape[0],0:pad.shape[1]]
    D=np.sqrt((yy-pad.shape[0]/2)**2+(xx-pad.shape[1]/2)**2)
    if kind=='ideal': H=(D<=d0).astype(float)
    elif kind=='butterworth': H=1/(1+(D/(d0+1e-9))**(2*order))
    else: H=np.exp(-(D**2)/(2*d0**2))
    if high: H=1-H
    out=np.fft.ifft2(np.fft.ifftshift(F*H)).real
    out=out[h//2:h//2+h,w//2:w//2+w]
    return np.uint8(np.clip(out,0,255))
im=pages[0][0]
outs=[freq_filter(im,k,35) for k in ['ideal','butterworth','gaussian']]
show_images([im]+outs,["Input","Ideal LPF","Butterworth LPF","Gaussian LPF"])
F=np.fft.fftshift(np.fft.fft2(im)); plt.figure(figsize=(5,4)); plt.imshow(np.log1p(np.abs(F)),cmap='gray'); plt.title("Log magnitude spectrum"); plt.axis('off'); plt.show()


## Task 6 – High-pass and homomorphic filtering

In [ ]:
def homomorphic(im,gamma_l=.65,gamma_h=1.5,c=1,d0=35):
    x=np.log1p(im.astype(np.float32)); h,w=x.shape
    F=np.fft.fftshift(np.fft.fft2(x)); yy,xx=np.mgrid[0:h,0:w]
    D2=(yy-h/2)**2+(xx-w/2)**2
    H=(gamma_h-gamma_l)*(1-np.exp(-c*D2/(d0*d0)))+gamma_l
    z=np.fft.ifft2(np.fft.ifftshift(F*H)).real
    z=np.exp(z); z=(z-z.min())/(z.max()-z.min()+1e-9)*255
    return np.uint8(np.clip(z,0,255))
im=pages[0][0]
outs=[freq_filter(im,'ideal',30,high=True),freq_filter(im,'butterworth',30,high=True),freq_filter(im,'gaussian',30,high=True),homomorphic(im)]
show_images([im]+outs,["Input","Ideal HPF","Butterworth HPF","Gaussian HPF","Homomorphic"],figsize=(15,4))

## Task 7 – Point, line and edge detection

In [ ]:
def point_line_edges(im):
    lap=cv2.Laplacian(im,cv2.CV_32F,ksize=3)
    points=(np.abs(lap)>90).astype(np.uint8)*255
    horiz=cv2.filter2D(im,-1,np.array([[-1,-1,-1],[2,2,2],[-1,-1,-1]],np.float32))
    vert=cv2.filter2D(im,-1,np.array([[-1,2,-1],[-1,2,-1],[-1,2,-1]],np.float32))
    lines=cv2.bitwise_or(cv2.threshold(horiz,180,255,cv2.THRESH_BINARY)[1],cv2.threshold(vert,180,255,cv2.THRESH_BINARY)[1])
    edges=cv2.Canny(cv2.GaussianBlur(im,(3,3),0.8),50,140)
    return points,lines,edges
im=pages[0][0]; p,l,e=point_line_edges(im)
show_images([im,p,l,e],["Input","Point response","Line response","Canny edges"])
def precision_recall(pred,truth):
    p=pred>0; t=truth>0; tp=np.logical_and(p,t).sum()
    fp=np.logical_and(p,~t).sum(); fn=np.logical_and(~p,t).sum()
    return tp/(tp+fp+1e-9),tp/(tp+fn+1e-9)
print("Illustrative pixel metrics against text mask (not defect annotations):",precision_recall(e,gt))

## Task 8 – Global and adaptive thresholding

In [ ]:
def iterative_threshold(im):
    t=float(im.mean())
    for _ in range(100):
        a=im[im<=t]; b=im[im>t]
        if len(a)==0 or len(b)==0: break
        nt=(float(a.mean())+float(b.mean()))/2
        if abs(nt-t)<0.5: break
        t=nt
    return cv2.threshold(im,int(t),255,cv2.THRESH_BINARY_INV)[1],int(t)
im,gt,_=pages[0]
it,T=iterative_threshold(im)
_,otsu=cv2.threshold(im,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
adapt=cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,11)
show_images([im,gt,it,otsu,adapt],["Input","Reference text","Iterative","Otsu","Adaptive"],figsize=(15,4))
display(pd.DataFrame([["Iterative",dice(it,gt),T],["Otsu",dice(otsu,gt),_],["Adaptive",dice(adapt,gt),"local"]],columns=["Method","Dice (synthetic reference)","Threshold"]))

## Task 9 – Region segmentation, page boundary and integrated pipeline

In [ ]:
def region_grow(im,seed,tol=22):
    # Simple 8-connected flood fill on a copy; output is the selected region.
    mask=np.zeros((im.shape[0]+2,im.shape[1]+2),np.uint8)
    work=im.copy()
    cv2.floodFill(work,mask,seed,255,loDiff=tol,upDiff=tol,flags=8|cv2.FLOODFILL_MASK_ONLY| (255<<8))
    return mask[1:-1,1:-1]
def page_boundary(im):
    ed=cv2.Canny(cv2.GaussianBlur(im,(5,5),0),40,120)
    contours,_=cv2.findContours(ed,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return ed,None
    c=max(contours,key=cv2.contourArea)
    return ed,cv2.boundingRect(c)
def scanner_pipeline(im):
    t0=time.perf_counter()
    enhanced=clahe(im,2.0,(8,8))
    den=cv2.medianBlur(enhanced,3)
    sharp=unsharp(den)
    binary=cv2.adaptiveThreshold(sharp,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,11)
    edges=cv2.Canny(cv2.GaussianBlur(enhanced,(5,5),0),40,120)
    contours,_=cv2.findContours(edges,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    box=cv2.boundingRect(max(contours,key=cv2.contourArea)) if contours else None
    return {"enhanced":enhanced,"binary":binary,"edges":edges,"box":box,"seconds":time.perf_counter()-t0}
res=scanner_pipeline(pages[0][0])
show_images([pages[0][0],res["enhanced"],res["binary"],res["edges"]],["Input","Enhanced","Text mask","Edges"])
print("Detected bounding rectangle:",res["box"],"Runtime seconds:",round(res["seconds"],4))


## Task 10 – RETAIN / MODIFY / REDESIGN on synthetic colour phone images

In [ ]:
def make_phone(seed=20,h=900,w=1200):
    g,mask,stain=make_document(seed,h,w,True,True)
    colour=cv2.cvtColor(g,cv2.COLOR_GRAY2BGR)
    # light-blue handwritten margin marks to test colour preservation
    cv2.putText(colour,"blue note",(int(w*.68),int(h*.92)),cv2.FONT_HERSHEY_SIMPLEX,1.0,(210,150,70),2,cv2.LINE_AA)
    # mild JPEG-like compression
    ok,enc=cv2.imencode('.jpg',colour,[cv2.IMWRITE_JPEG_QUALITY,82])
    return cv2.imdecode(enc,cv2.IMREAD_COLOR)
phone=make_phone()
def retain_strategy(bgr):
    gray=cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY); return scanner_pipeline(gray)
def modify_strategy(bgr):
    h,w=bgr.shape[:2]; scale=min(1.0,900/max(h,w))
    small=cv2.resize(bgr,None,fx=scale,fy=scale,interpolation=cv2.INTER_AREA)
    # Keep colour copy; blue pen is assessed in blue channel in OpenCV BGR.
    blue=small[:,:,0]
    return {"working":blue,"enhanced":clahe(blue,1.8,(8,8))}
def redesign_strategy(bgr):
    small=cv2.resize(bgr,(600,450),interpolation=cv2.INTER_AREA)
    gray=cv2.cvtColor(small,cv2.COLOR_BGR2GRAY)
    return {"working":homomorphic(gray),"edges":cv2.Canny(gray,50,140)}
strategies=[("RETAIN",retain_strategy),("MODIFY",modify_strategy),("REDESIGN",redesign_strategy)]
rows=[]
for name,fn in strategies:
    out,t=elapsed(fn,phone)
    rows.append([name,t*1000])
display(pd.DataFrame(rows,columns=["Strategy","Runtime (ms; this notebook environment)"]))
show_images([cv2.cvtColor(phone,cv2.COLOR_BGR2RGB),modify_strategy(phone)["working"],modify_strategy(phone)["enhanced"]],["Phone colour input","Blue channel","CLAHE blue channel"],figsize=(12,4))
print("Recommendation: MODIFY — downsample for analysis, retain original colour, preserve blue annotations, and perspective-correct the detected page.")


## Final notes for submission
1. Run **Runtime → Run all** in Google Colab. The notebook requires only OpenCV, NumPy, Matplotlib and Pandas, normally available in Colab.
2. Download the generated figures/tables if required by the instructor. All sample data is created internally.
3. Replace synthetic pages with at least 10 real pages for Tasks 1–9 and at least 15 colour phone photos for Task 10. Create reference text masks and defect/boundary annotations before claiming Dice, precision, recall or IoU as real-data measurements.
4. Pretrained DnCNN, document enhancement, HED, SAM 2, DIBCO binarization and OCR model comparisons require separately selected model weights and verified reference text. This notebook deliberately does not fabricate results for unavailable pretrained models.
5. Runtime varies with Colab hardware. For the required 3-second and 1-second limits, rerun on the specified four-core, 16 GB lab PC and record its hardware and measured timings.